In [1]:
# 导入工具包
from detection.datasets import pascal_voc
from detection.models.detectors import faster_rcnn
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

/opt/anaconda3/envs/dlcv/lib/python3.6/importlib/_bootstrap.py:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject
  return f(*args, **kwds)
/opt/anaconda3/envs/dlcv/lib/python3.6/importlib/_bootstrap.py:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject
  return f(*args, **kwds)
/opt/anaconda3/envs/dlcv/lib/python3.6/importlib/_bootstrap.py:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 192 from C header, got 216 from PyObject
  return f(*args, **kwds)


In [2]:
# 数据获取
train_dataset = pascal_voc.pascal_voc('train')
num_classes = len(train_dataset.classes)

In [3]:
# 加载模型
model = faster_rcnn.FasterRCNN(num_classes=num_classes)

In [4]:
# 优化器
optimizer = tf.keras.optimizers.SGD(1e-3,momentum=0.1,nesterov=True)

In [13]:
# 损失函数变化列表
loss_his = []

In [14]:
# 使用tf.gradientTape进行训练
# epoch
for epoch in range(2):
    # 获取索引
    indices = np.arange(train_dataset.num_gtlabels)
    # 打乱
    np.random.shuffle(indices)
    # 迭代次数
    iter = np.round(train_dataset.num_gtlabels/train_dataset.batch_size).astype(np.uint8)
    for idx in range(iter):
        # 获取batch数据索引
        idx = indices[idx]
        # 获取batch_size
        batch_image,batch_metas,batch_bboxes,batch_label = train_dataset[idx]
        # 梯度下降
        with tf.GradientTape() as tape:
            # 计算损失函数
            rpn_class_loss,rpn_bbox_loss,rcnn_class_loss,rcnn_bbox_loss = model((batch_image,batch_metas,batch_bboxes,batch_label),training=True)
            # 总损失
            loss = rpn_class_loss+rpn_bbox_loss+rcnn_class_loss+rcnn_bbox_loss
            # 计算梯度
            grads = tape.gradient(loss,model.trainable_variables)
            # 更新参数值
            optimizer.apply_gradients(zip(grads,model.trainable_variables))
            print("epoch:%d,batch:%d,loss:%f"%(epoch+1,idx,loss))
            loss_his.append(loss)

epoch:1,batch:441,loss:25.843674
epoch:1,batch:2660,loss:9.275081


KeyboardInterrupt: 

In [15]:
loss_his

[<tf.Tensor: shape=(), dtype=float32, numpy=25.843674>,
 <tf.Tensor: shape=(), dtype=float32, numpy=9.275081>]